# Chunking Strategy Comparison

Runs all three strategies from `src/chunking/` on a real document using the settings from `.env`.
Goal: understand chunk count, size distribution, and boundary quality before tuning parameters.

In [ ]:
import sys
sys.path.insert(0, "..")

import statistics
import textwrap

from src.ingestion import DocumentLoader
from src.embedding import Embedder
from src.chunking import FixedSizeChunker, RecursiveChunker, SemanticChunker
from configs.settings import AppConfig

cfg = AppConfig()
print(f"chunk_size                   : {cfg.chunk_size}")
print(f"chunk_overlap                : {cfg.chunk_overlap}")
print(f"semantic_breakpoint_threshold: {cfg.semantic_breakpoint_threshold}")
print(f"dense_embedding_model        : {cfg.dense_embedding_model}")

## 1. Load document

Using `corrective_rag.pdf` — 16 pages of dense technical prose and result tables.

In [2]:
loader = DocumentLoader()
doc    = loader.load("../data/raw/arxiv_papers/corrective_rag.pdf")

total_chars = sum(len(p.text) for p in doc.pages)
print(f"status : {doc.status.value}")
print(f"pages  : {doc.metadata.loaded_pages}/{doc.metadata.total_pages}")
print(f"total  : {total_chars:,} chars across all pages")

status : success
pages  : 16/16
total  : 65,744 chars across all pages


## 2. Run all three chunkers

In [3]:
fixed_chunker     = FixedSizeChunker()
recursive_chunker = RecursiveChunker()
embedder          = Embedder()
semantic_chunker  = SemanticChunker(embedder)   # reads threshold from AppConfig

fixed_chunks     = fixed_chunker.chunk([doc])
recursive_chunks = recursive_chunker.chunk([doc])
semantic_chunks  = semantic_chunker.chunk([doc])

print(f"fixed    : {len(fixed_chunks)} chunks")
print(f"recursive: {len(recursive_chunks)} chunks")
print(f"semantic : {len(semantic_chunks)} chunks")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   6%|▌         | 1/18 [00:00<00:07,  2.19it/s]

Batches:  11%|█         | 2/18 [00:00<00:04,  3.81it/s]

Batches:  17%|█▋        | 3/18 [00:00<00:02,  5.29it/s]

Batches:  22%|██▏       | 4/18 [00:00<00:02,  6.45it/s]

...

Batches:  61%|██████    | 11/18 [00:01<00:00, 15.72it/s]

Batches:  78%|███████▊  | 14/18 [00:01<00:00, 18.03it/s]

Batches:  94%|█████████▍| 17/18 [00:01<00:00, 20.87it/s]

Batches: 100%|██████████| 18/18 [00:01<00:00, 12.74it/s]

fixed    : 153 chunks
recursive: 194 chunks
semantic : 261 chunks


## 3. Comparison table

`mid_cut` = chunk whose last character is not sentence-ending punctuation — a proxy for how many chunks cut mid-sentence.

In [4]:
def table_row(label, chunks):
    sizes      = [c.chunk_size for c in chunks]
    mid_cut    = sum(1 for c in chunks if c.text.strip() and c.text.strip()[-1] not in ".!?")
    return dict(
        strategy = label,
        n        = len(chunks),
        min      = min(sizes),
        max      = max(sizes),
        avg      = round(statistics.mean(sizes)),
        std      = round(statistics.stdev(sizes) if len(sizes) > 1 else 0),
        mid_cut  = f"{mid_cut}/{len(chunks)}",
    )

rows = [
    table_row("fixed",     fixed_chunks),
    table_row("recursive", recursive_chunks),
    table_row("semantic",  semantic_chunks),
]

header = f"{'strategy':<12} {'n':>5} {'min':>6} {'max':>6} {'avg':>6} {'std':>6}  mid_cut"
print(header)
print("─" * len(header))
for r in rows:
    print(f"{r['strategy']:<12} {r['n']:>5} {r['min']:>6} {r['max']:>6} {r['avg']:>6} {r['std']:>6}  {r['mid_cut']}")

strategy         n    min    max    avg    std  mid_cut
───────────────────────────────────────────────────────
fixed          153     66    512    487     82  139/153
recursive      194     14    510    342    135  104/194
semantic       261      5    512    253    170  120/261


## 4. Example chunks — first three from each strategy

Same document, same position. Shows how boundary quality differs.

In [5]:
def show_first(chunks, label, n=3):
    print(f"\n{'━'*70}")
    print(f"  {label.upper()}  ({len(chunks)} chunks total, "
          f"avg {round(statistics.mean(c.chunk_size for c in chunks))} chars)")
    print(f"{'━'*70}")
    for i, chunk in enumerate(chunks[:n]):
        print(f"\n  ── chunk {i} ({chunk.chunk_size} chars) ──")
        print(textwrap.fill(chunk.text.strip(), width=68, initial_indent="  ", subsequent_indent="  "))
    if len(chunks) > n:
        print(f"\n  ... {len(chunks)-n} more not shown")

show_first(fixed_chunks,     "fixed-size")
show_first(recursive_chunks, "recursive")
show_first(semantic_chunks,  "semantic")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FIXED-SIZE  (153 chunks total, avg 487 chars)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── chunk 0 (512 chars) ──
  Corrective Retrieval Augmented Generation  Shi-Qi Yan 1 * , Jia-
  Chen Gu 2 * , Yun Zhu 3 , Zhen-Hua Ling 1  1 National Engineering
  Research Center of Speech and Language Information Processing,
  University of Science and Technology of China, Hefei, China 2
  Department of Computer Science, University of California, Los
  Angeles 3 Google DeepMind yansiki@mail.ustc.edu.cn , gujc@ucla.edu
  , yunzhu@google.com , zhling@ustc.edu.cn  Abstract  Large language
  models (LLMs) inevitably exhibit hallucinations since the acc

  ── chunk 1 (512 chars) ──
  ge models (LLMs) inevitably exhibit hallucinations since the
  accuracy of generated texts cannot be secured solely by the
  parametric knowledge they encapsulate. Although retrieval-
  augmented generation (RAG) is a p

## 5. Semantic threshold experiment

The `semantic_breakpoint_threshold` percentile controls how aggressively semantic chunking splits.
Higher percentile → fewer splits (only the most dissimilar sentence transitions break a chunk).

Here we sweep 70 → 80 → 90 → 95 on the same document to see the effect.

In [6]:
for pct in [70, 80, 90, 95]:
    ch = SemanticChunker(embedder, breakpoint_percentile=pct).chunk([doc])
    sizes = [c.chunk_size for c in ch]
    print(f"  threshold={pct:>2}th pct  →  "
          f"n={len(ch):>3}  "
          f"avg={round(statistics.mean(sizes)):>5} chars  "
          f"min={min(sizes):>4}  max={max(sizes):>5}")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   6%|▌         | 1/18 [00:00<00:04,  4.01it/s]

Batches:  17%|█▋        | 3/18 [00:00<00:01,  9.02it/s]

Batches:  28%|██▊       | 5/18 [00:00<00:01, 12.01it/s]

Batches:  44%|████▍     | 8/18 [00:00<00:00, 16.93it/s]

...

Batches:  28%|██▊       | 5/18 [00:00<00:01, 12.23it/s]

Batches:  44%|████▍     | 8/18 [00:00<00:00, 17.23it/s]

Batches:  67%|██████▋   | 12/18 [00:00<00:00, 23.65it/s]

Batches: 100%|██████████| 18/18 [00:00<00:00, 23.11it/s]

  threshold=95th pct  →  n=185  avg=  360 chars  min=   5  max=  512


## 6. Second document — narrative prose (`meditations.pdf`)

Repeat the comparison on long-form prose to see if patterns hold across genres.

In [7]:
doc2 = loader.load("../data/raw/gutenberg_books/meditations.pdf")
total2 = sum(len(p.text) for p in doc2.pages)
print(f"meditations.pdf: {doc2.metadata.loaded_pages} pages, {total2:,} chars total")

fixed2     = fixed_chunker.chunk([doc2])
recursive2 = recursive_chunker.chunk([doc2])
semantic2  = SemanticChunker(embedder).chunk([doc2])

rows2 = [
    table_row("fixed",     fixed2),
    table_row("recursive", recursive2),
    table_row("semantic",  semantic2),
]
print()
print(header)
print("─" * len(header))
for r in rows2:
    print(f"{r['strategy']:<12} {r['n']:>5} {r['min']:>6} {r['max']:>6} {r['avg']:>6} {r['std']:>6}  {r['mid_cut']}")

meditations.pdf: 155 pages, 463,969 chars total


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Batches:   1%|          | 1/124 [00:00<00:34,  3.61it/s]

Batches:   2%|▏         | 2/124 [00:00<00:24,  5.04it/s]

Batches:   2%|▏         | 3/124 [00:00<00:18,  6.43it/s]

...

Batches:  89%|████████▊ | 110/124 [00:05<00:00, 31.40it/s]

Batches:  93%|█████████▎| 115/124 [00:05<00:00, 35.61it/s]

Batches: 100%|██████████| 124/124 [00:05<00:00, 44.76it/s]

Batches: 100%|██████████| 124/124 [00:05<00:00, 23.35it/s]


strategy         n    min    max    avg    std  mid_cut
───────────────────────────────────────────────────────
fixed         1087     51    512    482     89  1019/1087
recursive     1287      5    512    367    130  1199/1287
semantic      1792      2    512    261    173  804/1792
